# ==========================================================
# Ethiopia Exchange Rate & Inflation Analysis 
# ==========================================================

In [5]:
# installing important packages
# !pip install requests
# !pip install pandas
# !pip install matplotlib

In [1]:
# ----------------------------
#  Import Libraries
# ----------------------------
import pandas as pd
import matplotlib.pyplot as plt
import requests

## 🌍 Exchange Rate & Inflation Analysis

### 💡 Objective
This project analyzes the relationship between **exchange rates** and **inflation** across selected countries using data from the World Bank API.

---

### 🌐 Data Source
- World Bank Open Data API  
- Indicators used:
  - **Exchange Rate** (`PA.NUS.FCRF`): Local currency per USD  
  - **Inflation** (`FP.CPI.TOTL.ZG`): Annual % change in consumer prices  

---

### 🌎 Countries Analyzed
- Ethiopia 🇪🇹  
- Kenya 🇰🇪  
- Japan 🇯🇵  
- Brazil 🇧🇷  
- India 🇮🇳  

---

### ⚙️ Methodology
- Retrieved historical data for both indicators using API requests  
- Cleaned and filtered out missing values  
- Merged datasets by year for each country  
- Organized data into a structured format using Pandas  

---

### 📊 Key Insight
- Exchange rates and inflation often show a **strong relationship**  
- Countries with higher inflation tend to experience **currency depreciation**  
- Trends vary by country depending on economic stability  

---

### 📈 Output
- Combined dataset: `exchange_and_inflation.csv`  
- Visualizations comparing exchange rates across countries  
- Ready for further analysis (correlation, regression, etc.)

---

### 🚀 Conclusion
This analysis demonstrates how macroeconomic indicators like inflation can influence exchange rates, providing useful insights for economic and financial decision-making.

In [ ]:
countries = {
    "Ethiopia": "ETH",
    "Kenya": "KEN",
    "Japan": "JPN",
    "Brazil": "BRA",
    "India": "IND"
}

exchange_indicator = "PA.NUS.FCRF"
inflation_indicator = "FP.CPI.TOTL.ZG"

all_data = []

for country, code in countries.items():
    # Exchange rates
    url_ex = f"https://api.worldbank.org/v2/country/{code}/indicator/{exchange_indicator}?format=json&per_page=1000"
    ex_data = requests.get(url_ex).json()[1]
    
    # Inflation
    url_inf = f"https://api.worldbank.org/v2/country/{code}/indicator/{inflation_indicator}?format=json&per_page=1000"
    inf_data = requests.get(url_inf).json()[1]
    
    # Merge by year
    for ex, inf in zip(ex_data[::-1], inf_data[::-1]):  # reverse to chronological
        if ex["value"] is not None and inf["value"] is not None:
            all_data.append({
                "Country": country,
                "Year": int(ex["date"]),
                "Exchange Rate": ex["value"],
                "Inflation (%)": inf["value"]
            })

df = pd.DataFrame(all_data)
df = df.sort_values(["Country", "Year"])
df.to_csv("exchange_and_inflation.csv", index=False)

print(df.head())

In [ ]:
plt.figure(figsize=(12,6))

for country in df["Country"].unique():
    subset = df[df["Country"] == country]
    
    plt.plot(subset["Year"], subset["Exchange Rate"], label=country)


plt.legend()
plt.title("Exchange Rate Comparison")
plt.xlabel("Year")
plt.ylabel("Local Currency per USD")
plt.grid(True)
plt.xticks(rotation=90)  # rotate labels so they are readable
plt.show()

In [ ]:
for country in df["Country"].unique():
    subset = df[df["Country"] == country]
    
    fig, ax1 = plt.subplots(figsize=(14,6))
    
    color = "tab:blue"
    ax1.set_xlabel("Year")
    ax1.set_ylabel("Exchange Rate", color=color)
    ax1.plot(subset["Year"], subset["Exchange Rate"], color=color, label="Exchange Rate")
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()  # second y-axis for inflation
    color = "tab:red"
    ax2.set_ylabel("Inflation (%)", color=color)
    ax2.plot(subset["Year"], subset["Inflation (%)"], color=color, label="Inflation")
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title(f"{country}: Exchange Rate vs Inflation")
    fig.tight_layout()
    plt.show()